# ReviewerBrain — Reviewer-Specific LoRA/QLoRA Training (Kaggle GPU)

Trains one QLoRA adapter per reviewer (`thockin`, `ezyang`) on the frozen SFT datasets, using the validated pipeline in this repository (`reviewerbrain.training.sft_data`).

**Status: this notebook has NOT been run yet.** No adapter, loss value, or evaluation result exists. When a run completes, record it in `docs/methodology/lora_training.md`.

**Setup (one-time, manual):**
1. Kaggle notebook settings: **GPU T4 x2** (or P100), **Internet ON**.
2. Attach a Kaggle Dataset that contains the cleaned corpora produced by the documented local commands (`scripts/data/clean_dataset.py` + `apply_clean_amendments.py` — deterministic, so they match the LFS-tracked evaluation artifacts):
   - `thockin_clean.jsonl`
   - `ezyang_clean.jsonl`
3. Set `CLEANED_DATA_SOURCE` below to that dataset's mount path under `/kaggle/input/`.

Nothing here hardcodes credentials or private dataset slugs; the only placeholder is the dataset path you attach.

In [ ]:
CLEANED_DATA_SOURCE = "<ATTACHED_KAGGLE_DATASET_PATH>"  # e.g. /kaggle/input/reviewerbrain-cleaned
REPO_URL = "https://github.com/SamarthGautam1/ReviewerBrain"
REVIEWERS = ["thockin", "ezyang"]          # keep in sync with configs/training/lora_qwen25coder7b.yaml
RUN_SMOKE_FIRST = True                      # tiny 16-example run before the real one

## 1. Environment check

In [ ]:
!nvidia-smi
import sys, torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.device_count(), "GPU(s)")

## 2. Repository + training dependencies

In [ ]:
%%bash
if [ ! -d ReviewerBrain ]; then git clone --depth 1 https://github.com/SamarthGautam1/ReviewerBrain.git; fi
cd ReviewerBrain
pip install -q -r requirements-training.txt

## 3. Cleaned corpora into the repo (SFT data is rebuilt on Kaggle — deterministic)

In [ ]:
import os, shutil
os.chdir("/kaggle/working/ReviewerBrain")
os.makedirs("data/processed", exist_ok=True)
for name in ("thockin_clean.jsonl", "ezyang_clean.jsonl"):
    src = os.path.join(CLEANED_DATA_SOURCE, name)
    assert os.path.exists(src), f"missing {src} — attach the cleaned-data dataset and set CLEANED_DATA_SOURCE"
    shutil.copy(src, f"data/processed/{name}")
    print("copied", name)
!wc -l data/processed/thockin_clean.jsonl data/processed/ezyang_clean.jsonl

## 4. Build the SFT datasets (frozen leakage guards apply here)
Held-out PRs are excluded by the frozen rule; `follow_up_patch` is never rendered; the split is PR-level and deterministic. The same command ran locally, so stats must match.

In [ ]:
!python scripts/training/build_sft_dataset.py
!cat data/processed/sft/stats.json

## 5. Pre-flight: `--prepare-only` (no model, no GPU cost)

In [ ]:
for reviewer in REVIEWERS:
    print(f"=== {reviewer} ===")
    !python scripts/training/train_lora.py --prepare-only --reviewer {reviewer}

## 6. Smoke run (optional but recommended)
Trains 16 examples × 1 epoch per reviewer to verify the full path (download → 4-bit load → LoRA → tokenize → step → save) before spending hours.

In [ ]:
if RUN_SMOKE_FIRST:
    for reviewer in REVIEWERS:
        !python scripts/training/train_lora.py --reviewer {reviewer} --smoke --out-root /kaggle/working/smoke_adapters

## 7. Real training (one adapter per reviewer)
Each run writes `adapters/<reviewer>/` with the LoRA adapter (never merged), tokenizer files and `training_report.json`.

In [ ]:
for reviewer in REVIEWERS:
    print(f"=== training {reviewer} ===", flush=True)
    !python scripts/training/train_lora.py --reviewer {reviewer} --out-root /kaggle/working/adapters

## 8. Inspect reports and package adapters

In [ ]:
import json, glob, shutil, os
for report_path in sorted(glob.glob("/kaggle/working/adapters/*/training_report.json")):
    with open(report_path) as f:
        r = json.load(f)
    final = [h for h in r["loss_history"] if "loss" in h]
    print(report_path)
    print("  base:", r["base_model"], "| seed:", r["seed"], "| merged:", r["merged"])
    print("  dataset:", {k: r["dataset"][k] for k in ("train_examples", "val_examples", "held_out_records")})
    print("  final loss:", final[-1]["loss"] if final else "n/a", "| versions:", r["package_versions"])

shutil.make_archive("/kaggle/working/reviewerbrain_adapters", "zip", "/kaggle/working", "adapters")
print("archive:", os.path.getsize("/kaggle/working/reviewerbrain_adapters.zip"), "bytes")

## 9. After training
- Download `reviewerbrain_adapters.zip`. Keep adapter binaries OUT of git (the repo ignores `adapters/`); place them at `<repo>/adapters/<reviewer>/` on whatever serving machine runs the four-mode pipeline (`scripts/inference/run_pipeline.py`, demo UI).
- Record the run (date, hardware, config SHA, final loss) in `docs/methodology/lora_training.md` **only after it actually happened** — do not pre-fill results.
- Generation evaluation on held-out queries is a separate step (`scripts/inference/run_pipeline.py` over the held-out artifact); it has NOT been performed.